DH演習2026 · 教材 第36版 · 2026-r36 · 2026-10-15

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第36版
教材ID: 2026-r36
教材日付: 2026-10-15
活動: text_counts
環境: Colab / Jupyter
ソース: 9177cc124aa68aa12bc35add1dd115cddc55cc55
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第7回：TEIから人物への言及を取り出して数える

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料は芥川龍之介『二人小町』第1場の発言14・15・19（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml)、CC0）です。人物のタグは授業で加えた注釈です。

## 1. 二つの注釈案を読む

`teacher`は教員の注釈案、`ai`はAIの注釈案です。固有名には`persName`、代名詞などの参照表現には`rs type="person"`を付け、`key`に人物IDを書いています。

In [ ]:
import xml.etree.ElementTree as ET

import pandas as pd

teacher = '''<body xmlns="http://www.tei-c.org/ns/1.0">
<sp who="#messenger"><speaker>使</speaker><p>いや、名前もあなたのように小町と云わなければいけないのです。</p></sp>
<sp who="#ono"><speaker>小町</speaker><p>小町！誰か小町と云う人はいなかったかしら。ああ、います。います。（発作的に笑い出しながら）<persName key="tama">玉造の小町</persName>と云う人がいます。<rs type="person" key="tama">あの人</rs>を代りにつれて行って下さい。</p></sp>
<sp who="#ono"><speaker>小町</speaker><p>（生き生きと）では<rs type="person" key="tama">あの人</rs>に行って貰って下さい。<rs type="person" key="tama">あの人</rs>はこの世にいるよりも、地獄に住みたいと云っています。誰も逢う人がいないものですから。</p></sp>
</body>'''

ai = '''<body xmlns="http://www.tei-c.org/ns/1.0">
<sp who="#messenger"><speaker>使</speaker><p>いや、名前も<rs type="person" key="ono">あなた</rs>のように小町と云わなければいけないのです。</p></sp>
<sp who="#ono"><speaker>小町</speaker><p>小町！誰か小町と云う人はいなかったかしら。ああ、います。います。（発作的に笑い出しながら）<persName key="tama">玉造の小町</persName>と云う人がいます。<rs type="person" key="tama">あの人</rs>を代りにつれて行って下さい。</p></sp>
</body>'''

## 2. タグの付いた言及を取り出す

`iter()`は要素を順にたどり、`itertext()`はタグの内側の文字を取り出します。`include_references`を`False`にすると`rs`を数えません。

実行前に、`if`の条件を根拠に、消える行の予測を説明してください。

In [ ]:
tei = teacher
include_references = True

T = "{http://www.tei-c.org/ns/1.0}"
mentions = []
for e in ET.fromstring(tei).iter():
    if e.tag == T + "persName" or (include_references and e.tag == T + "rs"):
        mentions.append({"text": "".join(e.itertext()), "key": e.get("key", "")})
pd.DataFrame(mentions)

## 3. 同じものをまとめて数える

`group_by`が`"text"`なら表記ごと、`"key"`なら人物IDごとに数えます。`counts.get(k, 0)`は、まだ数えていないキーなら0を返します。

In [ ]:
group_by = "text"

counts: dict[str, int] = {}
for m in mentions:
    counts[m[group_by]] = counts.get(m[group_by], 0) + 1
print(counts)
print("合計：", sum(counts.values()))

人物IDごとに数えると、「あの人」3件は`tama`に入ります。これは誰の、どの判断によるものでしょうか。元のタグに戻って確かめてください。

## 確認問題

AIの注釈案の言及を、参照表現を含めて人物ID別に数えてください。2.の`tei`を`ai`にし、3.の`group_by`を変えて再実行してください。`answer`に`ono`と`tama`の言及数を入れてください。

In [ ]:
answer = {"ono": None, "tama": None}

In [ ]:
#@title 答えを確かめる
if answer == {"ono": 1, "tama": 2}:
    print("結果：合格")
elif None in answer.values():
    print("結果：不合格（`answer`に二つの言及数を入れてください）")
elif answer == {"ono": 0, "tama": 1}:
    print("結果：不合格（rsを数えていません）")
else:
    print("結果：不合格（`key`ごとに、`persName`と`rs`の両方を数えてください）")